In [2]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

## 📊 DATA LOAD

Fact orders parquet dosyasını yükle ve kategori kolonunu belirle

In [3]:
df = pd.read_parquet("../data/processed/fact_orders.parquet")


# kategori kolonunu belirle
category_col = (
    "product_category_name_english"
    if "product_category_name_english" in df.columns
    else "product_category_name"
)

## 🔧 DATA PREPARATION

Sipariş-kategori ilişkisini hazırla, temizle ve çoklu kategori siparişlerini filtrele

In [4]:
df_rec = df[["order_id", category_col]].dropna().drop_duplicates()

df_rec[category_col] = (
    df_rec[category_col]
    .astype(str)
    .str.strip()
    .str.replace("_", " ")
    .str.title()
)


# sadece birden fazla kategori olan siparişler
order_counts = df_rec.groupby("order_id")[category_col].nunique()
valid_orders = order_counts[order_counts > 1].index

df_rec = df_rec[df_rec["order_id"].isin(valid_orders)]


# categorical dönüşüm
df_rec["order_id"] = df_rec["order_id"].astype("category")
df_rec[category_col] = df_rec[category_col].astype("category")

## 📐 SPARSE MATRIX

Kategori-sipariş ilişkisini sparse matrix formatında oluştur

In [5]:
row = df_rec[category_col].cat.codes
col = df_rec["order_id"].cat.codes

n_categories = len(df_rec[category_col].cat.categories)
n_orders = len(df_rec["order_id"].cat.categories)


category_order_matrix = csr_matrix(
    ([1] * len(df_rec), (row, col)),
    shape=(n_categories, n_orders)
)

## 🔗 COSINE SIMILARITY

Kategoriler arası benzerlik skorlarını hesapla (cosine similarity)

In [6]:
similarity = cosine_similarity(category_order_matrix, dense_output=False)

category_names = df_rec[category_col].cat.categories

## 🎯 RECOMMENDATION ENGINE

Benzer kategorileri bulan ana fonksiyon

In [7]:
def recommend_categories(category_name, top_n=5):

    if category_name not in category_names:
        return pd.DataFrame()

    idx = category_names.get_loc(category_name)

    sim_scores = similarity[idx].toarray().flatten()

    similar_idx = sim_scores.argsort()[::-1]


    recommendations = pd.DataFrame({
        "recommended_category": category_names[similar_idx],
        "similarity_score": sim_scores[similar_idx]
    })


    # kendisini çıkar
    recommendations = recommendations[
        recommendations["recommended_category"] != category_name
    ]


    # 0 similarity çıkar
    recommendations = recommendations[
        recommendations["similarity_score"] > 0
    ]


    # duplicate temizle
    recommendations = recommendations.drop_duplicates("recommended_category")


    return recommendations.head(top_n).reset_index(drop=True)

## 📝 TEXT OUTPUT

Dashboard'da gösterilecek metin formatında çıkış fonksiyonu

In [8]:
def recommendation_text(category_name, top_n=5):

    recs = recommend_categories(category_name, top_n=top_n)

    return {
        "selected_category": category_name,
        "title": f"Customers who bought {category_name} also bought:",
        "items": recs["recommended_category"].tolist()
    }

## ✅ TEST

Recommendation engine'i test et

In [9]:
result = recommendation_text("Auto", top_n=5)

print(result)

{'selected_category': 'Auto', 'title': 'Customers who bought Auto also bought:', 'items': ['Christmas Supplies', 'Construction Tools Construction', 'Computers Accessories', 'Home Comfort 2', 'Telephony']}
